# 🏅 Olympics AI — Natural Language to SQL with Fine-Tuned LLM

Fine-tuned **Llama 3.2-3B-Instruct** using **QLoRA** on 120 years of Olympic history.  
Ask any question in plain English → get real SQL → get real results from 271,116 athlete records.

**Tech stack:** `unsloth` · `trl` · `peft` · `bitsandbytes` · `transformers` · `streamlit` · `sqlite3`

---


## 1. Install Dependencies

In [ ]:
!pip install unsloth trl peft accelerate bitsandbytes datasets -q

## 2. Load & Explore the Olympic Dataset

We use the [120 Years of Olympic History](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results) dataset from Kaggle.  
Load it into a local SQLite database for fast SQL execution.


In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv("olympics.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nMedal value counts (NaN = no medal won):")
print(df["Medal"].value_counts(dropna=False))

conn = sqlite3.connect("olympics.db")
df.to_sql("athlete_events", conn, if_exists="replace", index=False)

count = pd.read_sql("SELECT COUNT(*) as n FROM athlete_events", conn).iloc[0]["n"]
print(f"\nLoaded {count:,} rows into olympics.db")
conn.close()

## 3. Explore the Database Schema & Data Quirks

Before fine-tuning, understand the exact values in the database.  
This is critical — the model must learn to use `Medal IS NOT NULL` (not `!= 'NA'`), `Team = 'United States'` (not `'USA'`), etc.


In [ ]:
conn = sqlite3.connect("olympics.db")

print("=== Medal values in SQLite ===")
print(pd.read_sql(
    "SELECT Medal, COUNT(*) as n FROM athlete_events GROUP BY Medal", conn))

print("\n=== How India is stored in Team column ===")
print(pd.read_sql(
    "SELECT DISTINCT Team FROM athlete_events WHERE Team LIKE '%India%'", conn))

print("\n=== Sample distinct sports ===")
print(pd.read_sql(
    "SELECT DISTINCT Sport FROM athlete_events ORDER BY Sport LIMIT 20", conn))

print("\n=== India medals ===")
print(pd.read_sql("""
    SELECT Name, Sport, Event, Medal, Year
    FROM athlete_events
    WHERE Team = 'India' AND Medal IS NOT NULL
    ORDER BY Year DESC
""", conn))

conn.close()

## 4. Define Schema & Build Training Dataset

We create 30 high-quality `(question → SQL)` pairs that cover the full range of query patterns:  
aggregations, filters, joins, date ranges, NULLs, GROUP BY, and more.

> **Key dataset facts the model must learn:**
> - `Medal IS NULL` for non-medal-winners (not `'NA'`)
> - `Sex` is `'M'` or `'F'`
> - `Team` is `'United States'`, `'Great Britain'` — not abbreviations


In [ ]:
from datasets import Dataset

SCHEMA = """CREATE TABLE athlete_events (
    ID INTEGER,
    Name VARCHAR,
    Sex VARCHAR,
    Age FLOAT,
    Height FLOAT,
    Weight FLOAT,
    Team VARCHAR,
    NOC VARCHAR,
    Games VARCHAR,
    Year INTEGER,
    Season VARCHAR,
    City VARCHAR,
    Sport VARCHAR,
    Event VARCHAR,
    Medal VARCHAR
)"""

qa_pairs = [
    ("Which country has won the most gold medals overall?",
     "SELECT Team, COUNT(*) as gold_count FROM athlete_events WHERE Medal='Gold' GROUP BY Team ORDER BY gold_count DESC LIMIT 5"),
    ("How many athletes competed in the 2016 Summer Olympics?",
     "SELECT COUNT(DISTINCT ID) as total_athletes FROM athlete_events WHERE Year=2016 AND Season='Summer'"),
    ("Show all medals won by India",
     "SELECT Name, Sport, Event, Medal, Year, City FROM athlete_events WHERE Team='India' AND Medal IS NOT NULL ORDER BY Year DESC"),
    ("Which athlete won the most gold medals?",
     "SELECT Name, Team, COUNT(*) as gold_medals FROM athlete_events WHERE Medal='Gold' GROUP BY Name, Team ORDER BY gold_medals DESC LIMIT 5"),
    ("How many gold medals has United States won in Swimming?",
     "SELECT COUNT(*) as gold_count FROM athlete_events WHERE Team='United States' AND Sport='Swimming' AND Medal='Gold'"),
    ("What is the average age of all Olympic athletes?",
     "SELECT ROUND(AVG(Age), 1) as avg_age FROM athlete_events WHERE Age IS NOT NULL"),
    ("Which female athletes won gold medals at the 2016 Olympics?",
     "SELECT Name, Team, Sport, Event FROM athlete_events WHERE Sex='F' AND Medal='Gold' AND Year=2016 AND Season='Summer'"),
    ("How many countries participated in the 2016 Olympics?",
     "SELECT COUNT(DISTINCT Team) as total_countries FROM athlete_events WHERE Year=2016 AND Season='Summer'"),
    ("Which city has hosted the Summer Olympics most often?",
     "SELECT City, COUNT(DISTINCT Year) as times_hosted FROM athlete_events WHERE Season='Summer' GROUP BY City ORDER BY times_hosted DESC LIMIT 5"),
    ("What is the average height of Basketball players?",
     "SELECT ROUND(AVG(Height), 1) as avg_height_cm FROM athlete_events WHERE Sport='Basketball' AND Height IS NOT NULL"),
    ("Show all sports in the 2016 Summer Olympics",
     "SELECT DISTINCT Sport FROM athlete_events WHERE Year=2016 AND Season='Summer' ORDER BY Sport"),
    ("Which country won the most medals at the 2008 Beijing Olympics?",
     "SELECT Team, COUNT(*) as medals FROM athlete_events WHERE Year=2008 AND Season='Summer' AND Medal IS NOT NULL GROUP BY Team ORDER BY medals DESC LIMIT 5"),
    ("How many medals did India win in the 2012 Olympics?",
     "SELECT Medal, COUNT(*) as count FROM athlete_events WHERE Team='India' AND Year=2012 AND Season='Summer' AND Medal IS NOT NULL GROUP BY Medal"),
    ("Who is the oldest gold medalist in Olympic history?",
     "SELECT Name, Team, Sport, Event, Age, Year FROM athlete_events WHERE Medal='Gold' AND Age IS NOT NULL ORDER BY Age DESC LIMIT 5"),
    ("Which sport has the most events?",
     "SELECT Sport, COUNT(DISTINCT Event) as num_events FROM athlete_events GROUP BY Sport ORDER BY num_events DESC LIMIT 10"),
    ("Show all Winter Olympics host cities",
     "SELECT DISTINCT City, Year FROM athlete_events WHERE Season='Winter' ORDER BY Year"),
    ("What percentage of athletes are female?",
     "SELECT ROUND(100.0 * SUM(CASE WHEN Sex='F' THEN 1 ELSE 0 END) / COUNT(*), 1) as female_pct FROM athlete_events"),
    ("Which athlete has participated in the most Olympics?",
     "SELECT Name, Team, COUNT(DISTINCT Year) as olympics_count FROM athlete_events GROUP BY Name, Team ORDER BY olympics_count DESC LIMIT 5"),
    ("How many total medals has Great Britain won?",
     "SELECT Medal, COUNT(*) as count FROM athlete_events WHERE Team='Great Britain' AND Medal IS NOT NULL GROUP BY Medal"),
    ("Show top 5 sports by gold medals at the 2016 Olympics",
     "SELECT Sport, COUNT(*) as gold_count FROM athlete_events WHERE Year=2016 AND Season='Summer' AND Medal='Gold' GROUP BY Sport ORDER BY gold_count DESC LIMIT 5"),
    ("What is the youngest age an athlete has won a gold medal?",
     "SELECT MIN(Age) as youngest_gold_age FROM athlete_events WHERE Medal='Gold' AND Age IS NOT NULL"),
    ("Which events did India win medals in at the 2008 Olympics?",
     "SELECT Name, Sport, Event, Medal FROM athlete_events WHERE Team='India' AND Year=2008 AND Medal IS NOT NULL"),
    ("How many unique sports have been part of the Olympics?",
     "SELECT COUNT(DISTINCT Sport) as total_sports FROM athlete_events"),
    ("Show total medals won by United States per year in Summer Olympics",
     "SELECT Year, COUNT(*) as medals FROM athlete_events WHERE Team='United States' AND Season='Summer' AND Medal IS NOT NULL GROUP BY Year ORDER BY Year"),
    ("Which country won the first ever Olympic gold medal?",
     "SELECT Name, Team, Event, Year, City FROM athlete_events WHERE Medal='Gold' ORDER BY Year ASC LIMIT 1"),
    ("How many athletes from India competed in all Olympics combined?",
     "SELECT COUNT(DISTINCT ID) as total_athletes FROM athlete_events WHERE Team='India'"),
    ("Which year had the highest number of participating athletes?",
     "SELECT Year, Season, COUNT(DISTINCT ID) as athletes FROM athlete_events GROUP BY Year, Season ORDER BY athletes DESC LIMIT 5"),
    ("Show all gold medals won by Jamaica",
     "SELECT Name, Sport, Event, Year, City FROM athlete_events WHERE Team='Jamaica' AND Medal='Gold' ORDER BY Year"),
    ("What is the average weight of weightlifting athletes?",
     "SELECT ROUND(AVG(Weight), 1) as avg_weight_kg FROM athlete_events WHERE Sport='Weightlifting' AND Weight IS NOT NULL"),
    ("How many medals has Kenya won in Athletics?",
     "SELECT Medal, COUNT(*) as count FROM athlete_events WHERE Team='Kenya' AND Sport='Athletics' AND Medal IS NOT NULL GROUP BY Medal"),
]

def make_training_text(question, sql):
    return f"""### Task
Convert the natural language question into a SQL query for the Olympics database.

### Database Schema
{SCHEMA}

### Question
{question}

### SQL Query
{sql}"""

train_data = Dataset.from_list([
    {"text": make_training_text(q, sql)} for q, sql in qa_pairs
])

print(f"Training examples: {len(train_data)}")
print("\nSample training example:")
print(train_data[0]["text"])

## 5. Load Base Model & Apply QLoRA Adapters

We use **Llama 3.2-3B-Instruct** loaded in **4-bit quantization** via `unsloth`.  
LoRA adapters (`r=16`) are added only to attention and MLP projection layers — only ~1% of weights are trained.


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()

## 6. Fine-Tune with SFTTrainer

Training config:
- **Epochs:** 10 (small dataset, needs more passes)
- **Learning rate:** 1e-4 with cosine scheduler + warmup
- **Batch size:** 2 × 2 gradient accumulation = effective batch of 4
- **Precision:** bf16 if supported, else fp16


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=10,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="olympics_model_outputs",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Time  : {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")
print(f"Loss  : {trainer_stats.metrics['train_loss']:.4f}")

## 7. SQL Generation & Execution Functions

Two helper functions:
- `generate_sql(question)` — prompts the fine-tuned model and extracts the SQL
- `run_sql(sql)` — executes it on the local SQLite database and returns a DataFrame
- `ask(question)` — combines both and prints results


In [ ]:
FastLanguageModel.for_inference(model)

SCHEMA_STR = """CREATE TABLE athlete_events (
    ID INTEGER, Name VARCHAR, Sex VARCHAR,
    Age FLOAT, Height FLOAT, Weight FLOAT,
    Team VARCHAR, NOC VARCHAR, Games VARCHAR,
    Year INTEGER, Season VARCHAR, City VARCHAR,
    Sport VARCHAR, Event VARCHAR, Medal VARCHAR
)"""

def generate_sql(question):
    prompt = f"""### Task
Convert the natural language question into a SQL query for the Olympics database.

### Database Schema
{SCHEMA_STR}

### Question
{question}

### SQL Query
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = full_text.split("### SQL Query")[-1].split("###")[0].strip()
    return sql.split(";")[0].strip() + ";"


def run_sql(sql):
    try:
        conn = sqlite3.connect("olympics.db")
        df = pd.read_sql_query(sql, conn)
        conn.close()
        return df, None
    except Exception as error:
        return None, str(error)


def ask(question):
    print(f"\n{'=' * 60}")
    print(f"Question : {question}")
    sql = generate_sql(question)
    print(f"SQL      : {sql}")
    df, error = run_sql(sql)
    if error:
        print(f"Error    : {error}")
        return None
    else:
        print(f"Result   : {len(df)} rows\n")
        print(df.to_string(index=False))
        return df

## 8. Test the Fine-Tuned Model

Run end-to-end: natural language → SQL → real data from 271,116 rows.


In [ ]:
ask("Which country has won the most gold medals overall?")
ask("Show all medals won by India")
ask("Who is the oldest gold medalist in Olympic history?")
ask("How many female athletes competed in the 2016 Olympics?")
ask("Which sport has the most events?")
ask("What is the average height of Basketball players?")
ask("Which city has hosted the Summer Olympics most often?")
ask("Which athlete has competed in the most Olympics?")
ask("Show all gold medals won by Jamaica")
ask("In which year did the most athletes compete?")

## 9. Push Fine-Tuned Model to HuggingFace Hub

Save the LoRA adapter weights and push to HuggingFace so the Streamlit app can load them.

> Replace `YOUR_HF_USERNAME` with your actual HuggingFace username before running.


In [ ]:
# Replace with your HuggingFace username
HF_USERNAME = "Abhishek9124"
REPO_NAME = f"{HF_USERNAME}/llama3-olympics-120years"

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"\nModel pushed to: https://huggingface.co/{REPO_NAME}")